# 🔬 BMRB Validation Pipeline: Benchmarking against Experimental NMR Data
### Programmatic retrieval and validation of synthetic protein models

---

## 🎯 Learning Objectives
In this tutorial, we demonstrate how to use the `bmrb_api` module to automate the validation of synthetic structures. You will learn to:
1. Fetch peer-reviewed experimental metadata from the **BMRB**.
2. Download experimental **Chemical Shifts** and compare them to synthetic predictions.
3. Retrieve **Distance Restraints (NOEs)** to verify the accuracy of a generated fold.
4. Query the **PDBe Validation API** for global geometric quality benchmarks.

In [ ]:
# 🔧 Environment Setup
import os
import sys

import matplotlib.pyplot as plt
import numpy as np

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q synth-pdb pynmrstar requests biotite matplotlib numpy
else:
    sys.path.append(os.path.abspath('../../'))

from synth_pdb.bmrb_api import BMRBAPI, PDBValidationAPI
from synth_pdb.chemical_shifts import predict_chemical_shifts
from synth_pdb.generator import PeptideGenerator

print("✅ Environment configured!")

## 1. Fetching BMRB Entry Metadata

We will focus on **Human Ubiquitin**, a small protein (76 aa) that is the gold standard for NMR method development. The BMRB Entry for Ubiquitin is **6457**.

In [ ]:
bmrb_id = "6457"

# Fetch entry metadata
metadata = BMRBAPI.get_entry_metadata(bmrb_id)
print(f"Title: {metadata.get('title')}")
authors = metadata.get('authors', [])
print(f"Authors: {', '.join([a['first_name'] + ' ' + a['last_name'] for a in authors[:3]])}... (and more)")

## 2. Comparing Chemical Shifts

Chemical shifts are sensitive to the local structural environment. We will generate a synthetic structure of Ubiquitin, predict its shifts, and compare them to the experimental assignments from BMRB 6457.

In [ ]:
# 1. Fetch experimental shifts
exp_shifts = BMRBAPI.fetch_chemical_shifts(bmrb_id)
print(f"✅ Fetched experimental shifts for {len(exp_shifts)} residues.")

# 2. Generate synthetic structure (Native sequence)
ubiq_seq = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"
gen = PeptideGenerator(ubiq_seq)
result = gen.generate(conformation="alpha") # Initial prediction as an alpha helix

# 3. Predict synthetic shifts
pred_shifts = predict_chemical_shifts(result.structure)
print("✅ Predicted synthetic shifts for 3D model.")

### Visualization: $C^\alpha$ Chemical Shift Correlation
We will plot the experimental vs. predicted $C^\alpha$ shifts. In a realistic model, these should correlate strongly ($R^2 > 0.9$).

In [ ]:
atom_type = "CA"
exp_vals = []
pred_vals = []

# Extract prediction for the first chain (usually 'A')
chain_id = list(pred_shifts.keys())[0] if pred_shifts else None
chain_preds = pred_shifts.get(chain_id, {}) if chain_id else {}

for res_id, atoms in exp_shifts.items():
    if res_id in chain_preds and atom_type in atoms and atom_type in chain_preds[res_id]:
        exp_vals.append(atoms[atom_type])
        pred_vals.append(chain_preds[res_id][atom_type])

if not exp_vals:
    print("⚠️ No overlapping chemical shifts found to plot!")
else:
    plt.figure(figsize=(7, 7))
    plt.scatter(exp_vals, pred_vals, alpha=0.6, edgecolors='k')
    plt.plot([min(exp_vals), max(exp_vals)], [min(exp_vals), max(exp_vals)], 'r--', alpha=0.8)
    plt.xlabel(f'Experimental {atom_type} Shift (ppm)')
    plt.ylabel(f'Predicted {atom_type} Shift (ppm)')
    plt.title(f'BMRB {bmrb_id} Validation: {atom_type} Correlation')
    plt.grid(True, alpha=0.3)

    # Calculate correlation
    corr = np.corrcoef(exp_vals, pred_vals)[0, 1]
    plt.text(50, 65, f'Correlation R = {corr:.3f}', fontsize=12, bbox={'facecolor': 'white', 'alpha': 0.8})
    plt.show()

## 3. Retrieving Distance Restraints (NOEs)

NOE restraints specify that certain atoms must be close in space. We can download these restraints and check if our model violates them.

In [ ]:
restraints = BMRBAPI.fetch_restraints(bmrb_id)
print(f"Retrieved {len(restraints)} experimental distance restraints.")

# Filter for long-range restraints (|i - j| > 10 residues)
long_range = [r for r in restraints if abs(r['index_1'] - r['index_2']) > 10]
print(f"Found {len(long_range)} long-range restraints defining the global fold.")

if long_range:
    r = long_range[0]
    print("\nExample Restraint:")
    print(f"  Residue {r['index_1']} ({r['atom_name_1']}) <-> Residue {r['index_2']} ({r['atom_name_2']})")
    print(f"  Max Distance: {r['upper_limit']} Å")

## 4. Geometric Quality Benchmarking

Finally, we can compare our model's quality to the entire PDB using the PDBe Validation API. We will use the high-resolution NMR structure **1D3Z** as our reference point.

In [ ]:
pdb_id = "1D3Z"
validation = PDBValidationAPI.get_validation_summary(pdb_id)

if validation:
    entry = validation[0]
    print(f"PDBe Validation for {pdb_id}:")
    print(f"  Clashscore Percentile: {entry.get('absolute_percentile_clashscore')}")
    print(f"  Ramachandran Percentile: {entry.get('absolute_percentile_ramachandran')}")
    print(f"  Sidechain Outliers: {entry.get('absolute_percentile_sidechain_outliers')}")

    print("\nInterpretation: A percentile of 90 means the structure is better than 90% of all PDB entries.")

## 5. Summary

The `bmrb_api` module provides a bridge between synthetic modeling and experimental reality. By programmatically fetching BMRB data, you can:
1. **Audit** structural models using independent experimental assignments.
2. **Benchmarking** structure prediction or refinement algorithms.
3. **Synthesize** large-scale datasets that are grounded in physical observations.